# 09b – Graz und die übrige Steiermark vergleichen

Dieses Notebook schätzt je ein Gründungs- und Überlebensmodell für Graz
(Gemeindekennziffer `60101`) und die übrige Steiermark im 100-Meter-Raster
für 2016–2025. Es verwendet die fertigen Modellstichproben aus Notebook 08
und die geprüften Schätzfunktionen aus Notebook 09.

Vor dem Ausführen werden folgende Dateien benötigt.

- `ANAL/data/100m/modelldaten/model_data_info.json`
- Die darin angegebenen Gründungs- und Überlebenstabellen.
- `ANAL/data/100m/raster_100m_styria.geoparquet` mit `grid_id` und `municipality_id`.
- `ANAL/09_estimate_and_save_models.ipynb` als Quelle der Modellfunktionen.

Die vorbereiteten Erreichbarkeitswerte einschließlich erreichbarer Standorte
jenseits der Gemeindegrenze gehen in die regionalen Schätzungen ein.
Alle acht Ergebnisdateien werden unter `RESULTS/graz_vs_rest/100m/final/`
gespeichert. Ein erneuter Lauf archiviert das bisherige regionale Ergebnisbündel
unter `ARCHIVE/results/graz_vs_rest/100m/`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import ast
import json
import platform
import shutil
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.duration.hazard_regression import PHReg

CONFIG = {
    "cell_size": 100, "run_name": "final",
    "maxiter_poisson": 300, "maxiter_nb2": 300, "maxiter_cox": 300,
    "alpha_lower": 1e-6, "alpha_upper": 1e4,
}
START_YEAR, END_YEAR, LAG_YEAR = 2016, 2025, 2015
REGIONS = {"graz": "Graz", "rest_styria": "Übrige Steiermark"}
PROJECT_DIR = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "ANAL").is_dir()
)
if CONFIG["cell_size"] != 100:
    raise ValueError("Notebook 9b verwendet das 100-Meter-Raster.")
CELL_SIZE, RESOLUTION = 100, "100m"
DATA_DIR = PROJECT_DIR / "ANAL" / "data" / RESOLUTION
PREPARED_DIR = DATA_DIR / "modelldaten"
RASTER_PATH = DATA_DIR / "raster_100m_styria.geoparquet"
NOTEBOOK09 = PROJECT_DIR / "ANAL" / "09_estimate_and_save_models.ipynb"
run_name = CONFIG["run_name"]
if not run_name or Path(run_name).name != run_name or run_name in {".", ".."}:
    raise ValueError("run_name muss ein einzelner Ordnername sein.")
RESULT_ROOT = (PROJECT_DIR / "RESULTS").resolve()
REGIONAL_RESULT_ROOT = (RESULT_ROOT / "graz_vs_rest").resolve()
BUNDLE_DIR = (REGIONAL_RESULT_ROOT / RESOLUTION / run_name).resolve()
if not BUNDLE_DIR.is_relative_to(REGIONAL_RESULT_ROOT) or not BUNDLE_DIR.is_relative_to(RESULT_ROOT):
    raise ValueError("Der Zielordner muss unter RESULTS/graz_vs_rest liegen.")


In [ ]:
# Die Auswahl lädt Definitionen anhand der stabilen Zell-IDs.
shared_cells = {
    "runtime-helpers": (
        {"note", "register_fit", "record_tests", "record_plot", "save_results"}, set()
    ),
    "load-prepared": (set(), {"FOUNDING_TERMS", "SURVIVAL_TERMS"}),
    "statistical-helpers": (
        {"design_matrix", "check_fit_arrays", "numerical_check", "count_estimate",
         "coefficient_table", "linear_effect"}, set()
    ),
    "hypothesis-definitions": (
        {"contrast_row", "hypothesis_table", "wald_row", "h4_joint"},
        {"POPULATION", "COUNT_TESTS", "SURVIVAL_TESTS", "H4_TERMS"}
    ),
    "cox-score-helpers": (
        {"cox_score_blocks", "verify_cox_cluster_implementation", "cox_newton"}, set()
    ),
}
source_notebook = json.loads(NOTEBOOK09.read_text(encoding="utf-8"))
for identifier, (function_names, constant_names) in shared_cells.items():
    cell = next((c for c in source_notebook["cells"] if c.get("id") == identifier), None)
    if cell is None:
        raise ValueError(f"Notebook 09 enthält die benötigte Zelle {identifier} nicht.")
    selected, found = [], set()
    for node in ast.parse("".join(cell["source"])).body:
        if isinstance(node, ast.FunctionDef) and node.name in function_names:
            selected.append(node)
            found.add(node.name)
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in constant_names:
                selected.append(node)
                found.add(target.id)
    missing = (function_names | constant_names) - found
    if missing:
        raise ValueError(f"Notebook 09, {identifier}, fehlende Definitionen {sorted(missing)}")
    module = ast.Module(body=selected, type_ignores=[])
    exec(compile(module, f"{NOTEBOOK09.name}#{identifier}", "exec"), globals())
print("Modellfunktionen aus Notebook 09 geladen.")


## 1. Modellstichproben und Gemeindezuordnung prüfen

Die Rasterzuordnung bestimmt die Region. Die Gemeindeangabe der vorbereiteten
Gründungstabelle dient als zusätzliche Konsistenzprüfung. Für das Überlebensmodell
wird geprüft, dass die vollständige Intervallfolge eines Standorts derselben
Region angehört. Jede Modellbeobachtung muss genau einer Teilmenge zugeordnet sein.


In [ ]:
def municipality_ids(values):
    result = values.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(5)
    if result.isna().any() or not result.str.fullmatch(r"\d{5}").all():
        raise ValueError("Fehlende oder ungültige Gemeindekennziffer.")
    return result


def assign_regions(frame, municipality_by_grid):
    municipalities = frame.grid_id.map(municipality_by_grid)
    if municipalities.isna().any():
        raise ValueError("Modellzellen ohne eindeutige Gemeindezuordnung.")
    if "municipality_id" in frame:
        if not municipality_ids(frame.municipality_id).eq(municipalities).all():
            raise ValueError("Gemeindezuordnung von Modelltabelle und Raster widerspricht sich.")
    result = frame.copy()
    result["region"] = np.where(municipalities.eq("60101"), "graz", "rest_styria")
    if "standort_id" in result:
        if result.groupby("standort_id").region.nunique().gt(1).any():
            raise ValueError("Die Intervalle eines Standorts verteilen sich auf beide Regionen.")
    return result


def load_region(model_type, region):
    ids = partition_ids[model_type][region]
    if not len(ids):
        raise ValueError(f"Leere Modellstichprobe für {model_type}, {region}.")
    frame = pd.read_parquet(
        table_paths[model_type], columns=model_columns[model_type],
        filters=[("grid_id", "in", ids.tolist())]
    )
    if len(frame) != partition_counts[model_type][region]:
        raise ValueError("Die regionale Auswahl hat Modellbeobachtungen verloren oder vervielfacht.")
    return frame.reset_index(drop=True)


In [ ]:
meta_path = PREPARED_DIR / "model_data_info.json"
if not meta_path.is_file():
    raise FileNotFoundError(f"Zuerst Notebook 08 ausführen. Modellmetadaten fehlen unter {meta_path}.")
prepared = json.loads(meta_path.read_text(encoding="utf-8"))
if (prepared.get("status") != "completed" or prepared.get("version") != "model-tables-3"
        or prepared.get("resolution_m") != CELL_SIZE):
    raise ValueError("Unpassendes Modelltabellenprofil. Notebook 08 erneut ausführen.")
if prepared.get("founding_terms") != FOUNDING_TERMS or prepared.get("survival_terms") != SURVIVAL_TERMS:
    raise ValueError("Die vorbereiteten Modellvariablen unterscheiden sich von Notebook 09.")

grid = pd.read_parquet(RASTER_PATH, columns=["grid_id", "municipality_id"])
if grid.grid_id.isna().any() or grid.grid_id.duplicated().any():
    raise ValueError("Das Raster muss jede grid_id eindeutig identifizieren.")
grid["municipality_id"] = municipality_ids(grid.municipality_id)
municipality_by_grid = grid.set_index("grid_id").municipality_id
model_columns = {
    "founding": ["grid_id", "municipality_id", "year", "quarter", "period", "births", *FOUNDING_TERMS],
    "survival": ["standort_id", "grid_id", "Fachgruppe_ID", "year", "period",
                 "start", "stop", "event", *SURVIVAL_TERMS],
}
key_columns = {
    "founding": ["grid_id", "period", "municipality_id", "year", "quarter"],
    "survival": ["standort_id", "grid_id", "period", "year"],
}
period_years = {f"{year}Q{quarter}": year for year in range(START_YEAR, END_YEAR + 1)
                for quarter in range(1, 5)}
period_quarters = {period: int(period[-1]) for period in period_years}
table_paths, partition_ids, partition_counts, partition_rows = {}, {}, {}, []
for model_type in model_columns:
    table_info = prepared["tables"][model_type]
    path = (PREPARED_DIR / table_info["file"]).resolve()
    if not path.is_relative_to(PREPARED_DIR.resolve()):
        raise ValueError("Modelltabellen müssen im vorbereiteten Modellordner liegen.")
    parquet = pq.ParquetFile(path)
    if parquet.metadata.num_rows != table_info["rows"] or not parquet.metadata.num_rows:
        raise ValueError(f"Ungültige Zeilenzahl der vorbereiteten Tabelle {model_type}.")
    if set(model_columns[model_type]) - set(parquet.schema_arrow.names):
        raise ValueError(f"Benötigte Modellspalten fehlen in {path.name}.")
    keys = pd.read_parquet(path, columns=key_columns[model_type])
    unique_keys = ["grid_id", "period"] if model_type == "founding" else ["standort_id", "period"]
    if keys.isna().any().any() or keys.duplicated(unique_keys).any():
        raise ValueError(f"Fehlende oder doppelte Modellschlüssel für {model_type}.")
    if not keys.period.isin(period_years).all() or not keys.year.eq(keys.period.map(period_years)).all():
        raise ValueError(f"Ungültige Modellperioden für {model_type}; erwartet werden 2016Q1 bis 2025Q4.")
    if model_type == "founding" and not keys.quarter.eq(keys.period.map(period_quarters)).all():
        raise ValueError("Quartal und Modellperiode widersprechen sich.")
    keys = assign_regions(keys, municipality_by_grid)
    table_paths[model_type] = path
    partition_ids[model_type], partition_counts[model_type] = {}, {}
    for region in REGIONS:
        subset = keys.loc[keys.region.eq(region)]
        partition_ids[model_type][region] = subset.grid_id.unique()
        partition_counts[model_type][region] = len(subset)
        partition_rows.append(dict(model=model_type, region=region, rows=len(subset),
                                   n_clusters=subset.grid_id.nunique()))
    if sum(partition_counts[model_type].values()) != len(keys):
        raise ValueError("Die regionalen Teilmengen decken die Modellstichprobe unvollständig ab.")
    if set(partition_ids[model_type]["graz"]) & set(partition_ids[model_type]["rest_styria"]):
        raise ValueError("Rasterzellen überschneiden sich zwischen den Regionen.")
    del keys, subset, parquet
del grid, municipality_by_grid
display(pd.DataFrame(partition_rows))


## 2. Regionale Hauptmodelle

Das Gründungsmodell verwendet NB2, absolute Gründungszahlen und vollständige
Jahr-Quartals-Effekte. In Graz entfällt `population_growth_yoy` gemäß der
gewählten regionalen Spezifikation. Notebook 08 definiert diese Variable als
`log1p(Zellbevölkerung aktuell) - log1p(Zellbevölkerung im Vorjahr)`.
Dieser Ausschluss ist eine bewusste Spezifikationsentscheidung.

Ein numerisch gescheitertes Grazer NB2-Modell erhält den bereits berechneten
Poisson-QMLE-Fit als Ersatz. Die Modellfamilie und der Anlass werden ausgewiesen.

Das Cox-Modell verwendet das Alter seit Gründung, verzögerten Eintritt,
Quartalsintervalle `(start, stop]`, Fachgruppenstrata und Efron-Bindungen.
Die Standardfehler beider Modellarten werden nach Rasterzellen geclustert.
Die numerische Prüfung der Cox-Clusterscores aus Notebook 09 wird ausgeführt.


In [ ]:
def describe_region(frame, model_type, region, model_id):
    terms = FOUNDING_TERMS if model_type == "founding" else SURVIVAL_TERMS
    support = frame[terms].agg(["min", "max", "mean", "std", "nunique"]).T
    support.index.name = "term"
    support = support.rename(columns={"nunique": "n_unique"}).reset_index()
    support["share_zero"] = frame[terms].eq(0).mean().reindex(support.term).to_numpy()
    support["status"] = np.where(support.n_unique.lt(2), "unidentified_constant", "diagnostic")
    coverage_frames[model_id + "_support"] = support.assign(
        model=model_id, region=region, check="covariate_support"
    )
    row = dict(
        model=model_id, region=region, stage="model_sample", rows=len(frame),
        units=frame.grid_id.nunique(), n_clusters=frame.grid_id.nunique(), excluded=0,
        events=int(frame.births.sum()) if model_type == "founding" else int(frame.event.sum()),
        share_with_stop=float(frame.pt_ohne_haltestelle.eq(0).mean()),
        share_without_stop=float(frame.pt_ohne_haltestelle.eq(1).mean()),
    )
    if model_type == "founding":
        row["zero_share"] = float(frame.births.eq(0).mean())
    else:
        locations = frame.groupby("standort_id").agg(entry=("start", "min"), event=("event", "max"))
        row.update(units=len(locations), right_censored=int(locations.event.eq(0).sum()),
                   left_truncated=int(locations.entry.gt(0).sum()))
    sample_rows.append(row)


In [ ]:
FIT_ERRORS = (RuntimeError, ValueError, np.linalg.LinAlgError)


def estimate_regional_count(frame, region):
    omitted = ["population_growth_yoy"] if region == "graz" else []
    terms = [term for term in FOUNDING_TERMS if term not in omitted]
    X, scaled, scale, constants = design_matrix(frame, terms, periods=True)
    del X
    y = frame.births.to_numpy(dtype=float)
    if not np.isfinite(y).all() or (y < 0).any() or (y != np.floor(y)).any() or y.sum() == 0:
        raise ValueError("Ungültige Gründungszahlen oder keine Gründungen in der Teilmenge.")
    if frame.grid_id.nunique() < 2:
        raise ValueError("Mindestens zwei Rastercluster werden benötigt.")
    poisson = count_estimate("poisson", scaled, y, frame.grid_id, scale)
    fallback_reason = ""
    try:
        fit = count_estimate("nb2", scaled, y, frame.grid_id, scale, poisson=poisson)
    except FIT_ERRORS as error:
        if region != "graz":
            raise
        fit = poisson
        fallback_reason = str(error)
        print(f"Graz verwendet Poisson-QMLE. NB2-Diagnose: {fallback_reason}")
    return dict(fit, excluded_terms=omitted, constant_terms=constants,
                fallback_reason=fallback_reason)


In [ ]:
def estimate_regional_cox(frame, region):
    X, scaled, scale, constants = design_matrix(frame, SURVIVAL_TERMS)
    names = list(X.columns)
    del X
    if frame.grid_id.nunique() < 2 or not frame.event.isin([0, 1]).all() or not frame.event.sum():
        raise ValueError("Unzureichende Rastercluster oder ungültige beziehungsweise fehlende Austritte.")
    if (not np.isfinite(frame[["start", "stop"]]).all().all()
            or frame.start.lt(0).any() or frame.stop.le(frame.start).any()
            or frame.Fachgruppe_ID.isna().any()):
        raise ValueError("Ungültige Cox-Intervalle oder fehlende Fachgruppenstrata.")
    ordered = frame.sort_values(["standort_id", "period"])
    previous_stop = ordered.groupby("standort_id").stop.shift()
    previous_event = ordered.groupby("standort_id").event.shift()
    if (previous_stop.notna() & ordered.start.ne(previous_stop)).any() or previous_event.eq(1).any():
        raise ValueError("Unterbrochene Standortverläufe oder Intervalle nach einem Austritt.")
    model = PHReg(
        frame.stop.to_numpy(dtype=float), scaled,
        status=frame.event.to_numpy(),
        entry=np.nextafter(frame.start.to_numpy(dtype=float), np.inf),
        strata=frame.Fachgruppe_ID.to_numpy(), ties="efron"
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        raw = cox_newton(model, label=f"Cox {REGIONS[region]}")
        params, bread = raw["params"], raw["bread"]
        scores, event_rows, schoenfeld, martingale, used = cox_score_blocks(model, params, frame.grid_id)
        if not np.allclose(scores.sum(axis=0), model.score(params), rtol=1e-7, atol=1e-6):
            raise RuntimeError("Die numerische Identität der Cox-Clusterscores ist verletzt.")
        covariance = bread @ (scores.T @ scores) @ bread
        check_fit_arrays(params, covariance, raw["loglike"])
    factor = scale.to_numpy()
    return dict(
        family="cox", names=names, params=(params / factor).tolist(),
        covariance=(covariance / np.outer(factor, factor)).tolist(),
        naive_covariance=(bread / np.outer(factor, factor)).tolist(),
        log_likelihood=raw["loglike"], aic=-2 * raw["loglike"] + 2 * len(params),
        converged=True, iterations=raw["iterations"], newton_step_se=raw["newton_step_se"],
        n_observations=len(frame), n_clusters=frame.grid_id.nunique(),
        n_events=int(frame.event.sum()), n_informative_intervals=int(used.sum()),
        warnings=sorted({str(w.message) for w in caught}),
        covariance_method="validated Efron case-weight grid-cell cluster sandwich CR0",
        ties="efron", entry_convention="(start, stop]; nextafter(start,+inf)",
        excluded_terms=[], constant_terms=constants, fallback_reason="",
    )


## 3. Prüfgrößen und regionale Unterschiede

H1 verwendet innerhalb jeder Region den Bevölkerungskontrast bei festem
Unternehmensbestand mit der vollständigen Kovarianz. Dazu kommen H2a, die
drei einzelnen H4-Effekte, der gemeinsame H5-Effekt sowie H2b und H2c.

Jede skalare Differenz wird als **Graz minus übrige Steiermark** auf der
Koeffizientenskala berechnet. Unter der Zellclusterannahme und bei
überschneidungsfreien Rasterclustern addieren sich die regionalen Varianzen.
Die Tabelle enthält 95-%-Konfidenzintervalle und zweiseitige p-Werte.
Exponentierte Differenzen entsprechen dem Verhältnis der regionalen IRR
beziehungsweise Hazard Ratios. H4 erhält zusätzlich einen gemeinsamen
Gleichheitstest mit der Summe beider Kovarianzmatrizen.

Konstante Variablen werden durch die vorhandene Designmatrix-Funktion erkannt.
Eine davon abhängige Prüfgröße erhält den Status `unidentified`.
Breite Intervalle bei geringer Streuung werden anhand der Streuungsdiagnostik
interpretiert. H5 behält die eingeschränkte Abdeckung des Hauptmodells bei.


In [ ]:
def regional_comparisons(fits):
    rows = []
    for model_type, definitions in [("founding", COUNT_TESTS), ("survival", SURVIVAL_TESTS)]:
        graz = fits.get(f"{model_type}_graz")
        rest = fits.get(f"{model_type}_rest_styria")
        for identifier, (weights, expected, hypothesis_coverage) in definitions.items():
            row = dict(
                model_type=model_type, region="graz_vs_rest", hypothesis_id=identifier,
                covariance="cell", coverage="unavailable", hypothesis_coverage=hypothesis_coverage,
                difference_direction="Graz minus übrige Steiermark",
                estimate=np.nan, std_error=np.nan, ci_lower=np.nan, ci_upper=np.nan, p_value=np.nan,
                effect_ratio=np.nan, ratio_ci_lower=np.nan, ratio_ci_upper=np.nan,
                ratio_label="Verhältnis der IRR" if model_type == "founding" else "Verhältnis der Hazard Ratios",
            )
            if graz is None or rest is None:
                row["detail"] = "Für den Vergleich werden beide regionalen Hauptmodelle benötigt."
            elif set(weights) - set(graz["names"]) or set(weights) - set(rest["names"]):
                row.update(coverage="unidentified", detail="Benötigte Variable in einer Region unbestimmt.")
            else:
                estimate_graz, se_graz = linear_effect(graz, weights)
                estimate_rest, se_rest = linear_effect(rest, weights)
                difference = estimate_graz - estimate_rest
                se = float(np.sqrt(se_graz**2 + se_rest**2))
                row.update(
                    coverage="tested", family_graz=graz["family"], family_rest=rest["family"],
                    estimate_graz=estimate_graz, std_error_graz=se_graz,
                    estimate_rest=estimate_rest, std_error_rest=se_rest,
                    estimate=difference, std_error=se,
                    ci_lower=difference - 1.96 * se, ci_upper=difference + 1.96 * se,
                    p_value=float(2 * stats.norm.sf(abs(difference / se))),
                )
                with np.errstate(over="ignore"):
                    row.update(
                        effect_graz=float(np.exp(estimate_graz)), effect_rest=float(np.exp(estimate_rest)),
                        effect_ratio=float(np.exp(difference)),
                        ratio_ci_lower=float(np.exp(row["ci_lower"])),
                        ratio_ci_upper=float(np.exp(row["ci_upper"])),
                    )
            rows.append(row)
    joint = dict(
        model_type="founding", region="graz_vs_rest", hypothesis_id="H4_joint",
        covariance="cell", coverage="unavailable",
        difference_direction="Graz minus übrige Steiermark",
    )
    graz, rest = fits.get("founding_graz"), fits.get("founding_rest_styria")
    if graz is None or rest is None:
        joint["detail"] = "Für den gemeinsamen H4-Vergleich werden beide Gründungsmodelle benötigt."
    elif set(H4_TERMS) - set(graz["names"]) or set(H4_TERMS) - set(rest["names"]):
        joint.update(coverage="unidentified", detail="Mindestens ein H4-Term ist regional unbestimmt.")
    else:
        graz_positions = [graz["names"].index(term) for term in H4_TERMS]
        rest_positions = [rest["names"].index(term) for term in H4_TERMS]
        difference = (np.asarray(graz["params"])[graz_positions]
                      - np.asarray(rest["params"])[rest_positions])
        covariance = (np.asarray(graz["covariance"])[np.ix_(graz_positions, graz_positions)]
                      + np.asarray(rest["covariance"])[np.ix_(rest_positions, rest_positions)])
        joint.update(wald_row("H4_joint", difference, covariance, np.eye(len(H4_TERMS)), "tested", "cell"))
        joint.update(family_graz=graz["family"], family_rest=rest["family"])
    rows.append(joint)
    return pd.DataFrame(rows)


In [ ]:
if BUNDLE_DIR.exists() and any(BUNDLE_DIR.iterdir()):
    archive_root = (PROJECT_DIR / "ARCHIVE").resolve()
    archive = archive_root / "results" / "graz_vs_rest" / RESOLUTION / (
        run_name + "_" + datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
    )
    if not archive.resolve().is_relative_to(archive_root):
        raise ValueError("Der Archivpfad muss unter ARCHIVE liegen.")
    archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(BUNDLE_DIR), str(archive))
    print("Vorheriger regionaler Lauf archiviert unter", archive)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

sample_rows = [dict(row, region="all_styria", model="prepared_" + row["model"])
               for row in prepared.get("sample_description", [])]
diagnostic_rows = [dict(row, region="all_styria", model="prepared_" + row["model"])
                   for row in prepared.get("diagnostics", [])]
coefficient_frames, hypothesis_frames, covariance_frames, plot_frames = {}, {}, {}, {}
fit_summaries, coverage_frames, regional_fits = {}, {}, {}
model_regions = {f"{kind}_{region}": region for region in REGIONS for kind in model_columns}
model_states = {
    model: dict(status="pending", region=region) for model, region in model_regions.items()
}
manifest = dict(
    status="running", pipeline_version="09b-regional-1", resolution_m=CELL_SIZE,
    run_name=run_name, created_at=datetime.now(timezone.utc).isoformat(), config=CONFIG,
    regions=REGIONS, graz_municipality_id="60101", models=model_states,
    study_period=dict(start_year=START_YEAR, end_year=END_YEAR, lead_in_year=LAG_YEAR),
    time_aggregation=dict(founding="quarter", survival="quarter"),
    software=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                  scipy=scipy.__version__, statsmodels=statsmodels.__version__),
    sources=[str(p.relative_to(PROJECT_DIR)) for p in [meta_path, RASTER_PATH, NOTEBOOK09, *table_paths.values()]],
    prepared_sources=prepared.get("source_files", []),
    event_definition=prepared.get("event_definition", {}),
    covariance_method="grid-cell clusters; disjoint regional clusters",
    founding_graz_excluded_terms=["population_growth_yoy"],
    population_growth_definition="log1p(current cell population) - log1p(previous-year cell population)",
    comparison_direction="Graz minus übrige Steiermark",
    table_note=(
        "Graz ohne population_growth_yoy gemäß gewählter Spezifikation. "
        "Die Variable verwendet die Veränderung der log1p-Zellbevölkerung. "
        "Erreichbarkeitswerte einschließlich gemeindeübergreifender Einzugsgebiete. "
        "95-%-Konfidenzintervalle mit Zellclusterung. Geringe Streuung und breite Intervalle "
        "anhand der regionalen Unterstützung beurteilen. "
        "H5 bildet den gemeinsamen Autobahneffekt mit eingeschränkter Hypothesenabdeckung ab."
    ),
)


In [ ]:
def save_regional_results(status="running"):
    comparison = regional_comparisons(regional_fits)
    record_plot("regional_comparison", comparison, "regional_comparison")
    target = BUNDLE_DIR / "regionalvergleiche.csv"
    temporary = target.with_name(target.name + ".tmp")
    comparison.to_csv(temporary, index=False)
    temporary.replace(target)
    for row in diagnostic_rows:
        row.setdefault("region", model_regions.get(row["model"], "all_styria"))
    save_results(status)
    manifest["outputs"].append("regionalvergleiche.csv")
    temporary = BUNDLE_DIR / "laufinfo.json.tmp"
    temporary.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False,
                   default=lambda value: value.item() if hasattr(value, "item") else str(value)),
        encoding="utf-8"
    )
    temporary.replace(BUNDLE_DIR / "laufinfo.json")


try:
    library_covariance_valid = verify_cox_cluster_implementation()
    note("cox_validation", "cluster_score_validation", "OK",
         f"Finite-Differenzen-Prüfung bestanden; Bibliotheksvergleich={library_covariance_valid}")
    cox_validation_error = ""
except FIT_ERRORS as error:
    cox_validation_error = str(error)
    note("cox_validation", "cluster_score_validation", "FAILED", cox_validation_error)
save_regional_results()


## 4. Vier Hauptschätzungen ausführen

Die beiden Modellarten werden für jede Region nacheinander geladen und geschätzt.
Die Ergebnisdateien werden nach jedem Modell aktualisiert. Numerische Modellfehler
werden einzeln protokolliert. Erfolgreiche Schätzungen bleiben als Zwischenstand
verfügbar. `laufinfo.json` enthält Modellfamilie, ausgeschlossene und konstante
Variablen, einen möglichen Poisson-Fallback sowie den Abschlussstatus.


In [ ]:
for region, region_label in REGIONS.items():
    for model_type in model_columns:
        model_id = f"{model_type}_{region}"
        print(f"\n{region_label} – {model_type}", flush=True)
        model_states[model_id]["status"] = "running"
        save_regional_results()
        frame = None
        try:
            frame = load_region(model_type, region)
            describe_region(frame, model_type, region, model_id)
            if model_type == "founding":
                fit = estimate_regional_count(frame, region)
                tests = pd.concat(
                    [hypothesis_table(fit, COUNT_TESTS), pd.DataFrame([h4_joint(fit)])],
                    ignore_index=True
                )
                effect = "irr"
            else:
                if cox_validation_error:
                    raise RuntimeError(f"Cox-Clusterscore-Prüfung fehlgeschlagen: {cox_validation_error}")
                fit = estimate_regional_cox(frame, region)
                tests = hypothesis_table(fit, SURVIVAL_TESTS)
                effect = "hazard_ratio"
            register_fit(model_id, fit, group=region, effect=effect)
            for frames in [coefficient_frames, covariance_frames]:
                frames[model_id]["region"] = region
            fit_summaries[model_id]["region"] = region
            record_tests(model_id, tests.assign(region=region, family=fit["family"]))
            regional_fits[model_id] = fit
            model_states[model_id].update(
                status="estimated", family=fit["family"], excluded_terms=fit["excluded_terms"],
                constant_terms=fit["constant_terms"], fallback_reason=fit["fallback_reason"],
                covariance_method=fit["covariance_method"],
            )
            note(model_id, "excluded_terms", "INFO", json.dumps(fit["excluded_terms"]))
            note(model_id, "constant_terms", "INFO", json.dumps(fit["constant_terms"]))
            if fit["fallback_reason"]:
                note(model_id, "poisson_fallback", "FALLBACK", fit["fallback_reason"])
            display(tests)
        except FIT_ERRORS as error:
            model_states[model_id].update(status="failed", detail=str(error))
            note(model_id, "main_fit", "FAILED", str(error))
            print(f"{model_id} fehlgeschlagen: {error}", flush=True)
        finally:
            del frame
        save_regional_results()

estimated = sum(state["status"] == "estimated" for state in model_states.values())
final_status = "completed" if estimated == 4 else "partially_completed" if estimated else "failed"
save_regional_results(final_status)


In [ ]:
display(pd.DataFrame(model_states).T.rename_axis("model").reset_index())
display(pd.DataFrame(sample_rows).loc[lambda table: table.region.isin(REGIONS)])
comparison = regional_comparisons(regional_fits)
display(comparison)
support_tables = [table for key, table in coverage_frames.items() if key.endswith("_support")]
if support_tables:
    display(pd.concat(support_tables, ignore_index=True))
print(manifest["table_note"])
print(f"{estimated}/4 Hauptmodelle geschätzt. Status {final_status}. Ergebnisordner {BUNDLE_DIR}")
